In [ ]:
from config import Config
from data import ImagePairDataset
import torch
from torchvision.transforms import v2 as T
from torchvision.transforms.functional import to_pil_image
from typing import Optional

In [ ]:
def generate_transform(config: Config) -> T.Transform:

    def jitter_to_range(jitter: Optional[float], base: float = 0.0) -> tuple[float, float]:
        if jitter is None:
            return None
        return (base - jitter, base + jitter)

    transforms = []

    transforms.append(T.ToImage())
    transforms.append(T.ToDtype(torch.float32, scale=True))

    transforms.append(T.RandomAffine(degrees=config.TRANSFORM_ROTATION or 0,
                                    translate=jitter_to_range(config.TRANSFORM_TRANSLATE),
                                    scale=jitter_to_range(config.TRANSFORM_SCALE, base=1.0)))

    if config.TRANSFORM_FLIP == True:
        transforms.append(T.RandomHorizontalFlip())

    transforms.append(T.ColorJitter(brightness=config.TRANSFORM_BRIGHTNESS,
                     contrast=config.TRANSFORM_CONTRAST,
                     saturation=config.TRANSFORM_SATURATION,
                     hue=config.TRANSFORM_HUE))

    if config.TRANSFORM_BLUR is not None:
        transforms.append(T.GaussianBlur(kernel_size=5, sigma=(0.0, config.TRANSFORM_BLUR)))

    if config.TRANSFORM_NOISE is not None:
        transforms.append(T.GaussianNoise(mean=0, sigma=config.TRANSFORM_NOISE))

    return T.Compose(transforms)

In [ ]:
config = Config()
transform = generate_transform(config)
data = ImagePairDataset(split='train', transform=transform)

first, second, target = data[0]
display(to_pil_image(first))
display(to_pil_image(second))
print('Equivalent' if target == 1.0 else 'Different')